In [1]:
import pandas as pd

## Investigating and fixing the train.csv

In [2]:
train = pd.read_csv("train.csv")

train.head()

/tmp/ipykernel_2030/2870560131.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("train.csv")


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [3]:
test = pd.read_csv("test.csv")

test.head()

,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
0,1,1,4,2015-09-17,1.0,1,0,0
1,2,3,4,2015-09-17,1.0,1,0,0
2,3,7,4,2015-09-17,1.0,1,0,0
3,4,8,4,2015-09-17,1.0,1,0,0
4,5,9,4,2015-09-17,1.0,1,0,0


In [4]:
store = pd.read_csv("store.csv")

store.head()

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [5]:
print(train.shape)
print(test.shape)
print(store.shape)

(1017209, 9)
(41088, 8)
(1115, 10)


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype 
---  ------         --------------    ----- 
 0   Store          1017209 non-null  int64 
 1   DayOfWeek      1017209 non-null  int64 
 2   Date           1017209 non-null  object
 3   Sales          1017209 non-null  int64 
 4   Customers      1017209 non-null  int64 
 5   Open           1017209 non-null  int64 
 6   Promo          1017209 non-null  int64 
 7   StateHoliday   1017209 non-null  object
 8   SchoolHoliday  1017209 non-null  int64 
dtypes: int64(7), object(2)
memory usage: 69.8+ MB


In [7]:
# Since each column has a non-null value, me might want to check ourselves before moving further to the EDA.

In [8]:
train['StateHoliday'].unique()

array(['0', 'a', 'b', 'c', 0], dtype=object)

As per the Kaggle dataset rule, these values "a", "b", "c" means this:

a = public holiday, b = Easter holiday, c = Christmas

In [9]:
train['StateHoliday'].value_counts()

,count
StateHoliday,
0,855087
0,131072
a,20260
b,6690
c,4100


In [10]:
train.groupby('Open')['Sales'].describe()

,count,mean,std,min,25%,50%,75%,max
Open,,,,,,,,
0,172817.0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0
1,844392.0,6955.514291,3104.21468,0.0,4859.0,6369.0,8360.0,41551.0


When the store was Open, the sales were still somehow at one point reached to a minimum value of 0. We have to check that

In [11]:
# Filter to rows where the store was marked Open but Sales is exactly zero
zero_sales_open = train[(train['Open'] == 1) & (train['Sales'] == 0)]

# Count how many such rows exist — the scale tells us whether this is a rare edge case or a systemic issue
print(len(zero_sales_open))

54


In [12]:
# Inspect actual rows (Customers included) to see if zero sales came with zero foot traffic too, or looks like a real error
zero_sales_open[['Store','Date','DayOfWeek','Sales','Customers','Promo','StateHoliday','SchoolHoliday']].head(10)

,Store,Date,DayOfWeek,Sales,Customers,Promo,StateHoliday,SchoolHoliday
86825,971,2015-05-15,5,0,0,0,0,1
142278,674,2015-03-26,4,0,0,0,0,0
196938,699,2015-02-05,4,0,0,1,0,0
322053,708,2014-10-01,3,0,0,1,0,0
330176,357,2014-09-22,1,0,0,0,0,0
340348,227,2014-09-11,4,0,0,0,0,0
340860,835,2014-09-11,4,0,0,0,0,0
341795,835,2014-09-10,3,0,0,0,0,0
346232,548,2014-09-05,5,0,0,1,0,1
346734,28,2014-09-04,4,0,0,1,0,0


In [13]:
# Since the Date column is an object dtype, we will convert it to DateTime

train['Date'] = pd.to_datetime(train['Date'])

train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype         
---  ------         --------------    -----         
 0   Store          1017209 non-null  int64         
 1   DayOfWeek      1017209 non-null  int64         
 2   Date           1017209 non-null  datetime64[ns]
 3   Sales          1017209 non-null  int64         
 4   Customers      1017209 non-null  int64         
 5   Open           1017209 non-null  int64         
 6   Promo          1017209 non-null  int64         
 7   StateHoliday   1017209 non-null  object        
 8   SchoolHoliday  1017209 non-null  int64         
dtypes: datetime64[ns](1), int64(7), object(1)
memory usage: 69.8+ MB


In [14]:
date_range_per_store = train.groupby('Store')['Date'].agg(['min', 'max', 'count'])  # first date, last date, and how many rows each store actually has

date_range_per_store

,min,max,count
Store,,,
1,2013-01-01,2015-07-31,942
2,2013-01-01,2015-07-31,942
3,2013-01-01,2015-07-31,942
4,2013-01-01,2015-07-31,942
5,2013-01-01,2015-07-31,942
...,...,...,...
1111,2013-01-01,2015-07-31,942
1112,2013-01-01,2015-07-31,942
1113,2013-01-01,2015-07-31,942


In [15]:
date_range_per_store['expected_days'] = (date_range_per_store['max'] - date_range_per_store['min']).dt.days + 1  # how many calendar days should exist between first and last date

date_range_per_store['missing_days'] = date_range_per_store['expected_days'] - date_range_per_store['count']  # difference = potential gaps

In [16]:
date_range_per_store['missing_days'].value_counts().sort_index()

,count
missing_days,
0,935
184,180


935 vs 180 stores: 935 stores have zero gaps (clean data); 180 stores are all missing the exact same 184 days — that exact match across 180 stores means one shared event (like a mass closure), not random errors.

We'll check this by the following:

In [17]:
gap_stores = date_range_per_store[date_range_per_store['missing_days'] == 184].index  # get the list of the 180 affected store IDs

In [18]:
sample_store = gap_stores[54]  # pick one to inspect closely

store_data = train[train['Store'] == sample_store].sort_values('Date')  # get all rows for that store, in date order

store_data['date_diff'] = store_data['Date'].diff()  # compute the gap between each row and the previous one

In [19]:
store_data[store_data['date_diff'] > pd.Timedelta(days=1)]  # show us exactly where the gap starts

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,date_diff
235562,298,4,2015-01-01,0,0,0,0,a,1,185 days


In [20]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype         
---  ------         --------------    -----         
 0   Store          1017209 non-null  int64         
 1   DayOfWeek      1017209 non-null  int64         
 2   Date           1017209 non-null  datetime64[ns]
 3   Sales          1017209 non-null  int64         
 4   Customers      1017209 non-null  int64         
 5   Open           1017209 non-null  int64         
 6   Promo          1017209 non-null  int64         
 7   StateHoliday   1017209 non-null  object        
 8   SchoolHoliday  1017209 non-null  int64         
dtypes: datetime64[ns](1), int64(7), object(1)
memory usage: 69.8+ MB


In [21]:
train['StateHoliday'].unique()

array(['0', 'a', 'b', 'c', 0], dtype=object)

In [22]:
# In the StateHoliday column, there are values such as 'a' 'b' 'c', and 0 is written as '0' in some places, so we need to fix it

In [23]:
# Cast StateHoliday to string dtype, collapsing the integer-0 and string-'0' rows into one consistent category

train['StateHoliday'] = train['StateHoliday'].astype(str)

In [24]:
# Confirm we now have exactly 4 clean categories instead of 5

print(train['StateHoliday'].unique())

print()

print(train['StateHoliday'].value_counts())

['0' 'a' 'b' 'c']

StateHoliday
0    986159
a     20260
b      6690
c      4100
Name: count, dtype: int64


In [25]:
# Confirm no rows were lost or gained in the process
print(train.shape)

(1017209, 9)


## Investigating and fixing the test.csv

In [26]:
test.head()

,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
0,1,1,4,2015-09-17,1.0,1,0,0
1,2,3,4,2015-09-17,1.0,1,0,0
2,3,7,4,2015-09-17,1.0,1,0,0
3,4,8,4,2015-09-17,1.0,1,0,0
4,5,9,4,2015-09-17,1.0,1,0,0


In [27]:
test.info()

print()

test.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41088 entries, 0 to 41087
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             41088 non-null  int64  
 1   Store          41088 non-null  int64  
 2   DayOfWeek      41088 non-null  int64  
 3   Date           41088 non-null  object 
 4   Open           41077 non-null  float64
 5   Promo          41088 non-null  int64  
 6   StateHoliday   41088 non-null  object 
 7   SchoolHoliday  41088 non-null  int64  
dtypes: float64(1), int64(5), object(2)
memory usage: 2.5+ MB



,0
Id,0
Store,0
DayOfWeek,0
Date,0
Open,11
Promo,0
StateHoliday,0
SchoolHoliday,0


After checking it seems that the Open column in test.csv contains 11 nulls values. Open is an important feature since it basically mentions whether the store was trading / open or not, so we need to check into it.

In [28]:
# Isolate the rows where Open is missing to see if there's a pattern (same store? same date range?)
test[test['Open'].isna()]

,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
479,480,622,4,2015-09-17,NaN,1,0,0
1335,1336,622,3,2015-09-16,NaN,1,0,0
2191,2192,622,2,2015-09-15,NaN,1,0,0
3047,3048,622,1,2015-09-14,NaN,1,0,0
4759,4760,622,6,2015-09-12,NaN,0,0,0
5615,5616,622,5,2015-09-11,NaN,0,0,0
6471,6472,622,4,2015-09-10,NaN,0,0,0
7327,7328,622,3,2015-09-09,NaN,0,0,0
8183,8184,622,2,2015-09-08,NaN,0,0,0
9039,9040,622,1,2015-09-07,NaN,0,0,0


In [29]:
# Since train.csv contained those type of values, we need to check test.csv's StateHoliday values before touching anything — confirm whether the same mixed-type issue exists here

print(test['StateHoliday'].unique())
print(test['StateHoliday'].value_counts())

['0' 'a']
StateHoliday
0    40908
a      180
Name: count, dtype: int64


Since those days on which the data is missing, those were working days, not weekends, and as per the most widely-used approach so we will be implementing the same approach

In [30]:
test['Open'] = test['Open'].fillna(1)

In [31]:
test.isnull().sum()

,0
Id,0
Store,0
DayOfWeek,0
Date,0
Open,0
Promo,0
StateHoliday,0
SchoolHoliday,0


In [32]:
# The StateHoliday column contains str values but is labeled as object, lets change it:

test['StateHoliday'] = test['StateHoliday'].astype(str)

In [33]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41088 entries, 0 to 41087
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             41088 non-null  int64  
 1   Store          41088 non-null  int64  
 2   DayOfWeek      41088 non-null  int64  
 3   Date           41088 non-null  object 
 4   Open           41088 non-null  float64
 5   Promo          41088 non-null  int64  
 6   StateHoliday   41088 non-null  object 
 7   SchoolHoliday  41088 non-null  int64  
dtypes: float64(1), int64(5), object(2)
memory usage: 2.5+ MB


## Investigating and fixing store.csv

In [34]:
store.head()

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [35]:
store.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Store                      1115 non-null   int64  
 1   StoreType                  1115 non-null   object 
 2   Assortment                 1115 non-null   object 
 3   CompetitionDistance        1112 non-null   float64
 4   CompetitionOpenSinceMonth  761 non-null    float64
 5   CompetitionOpenSinceYear   761 non-null    float64
 6   Promo2                     1115 non-null   int64  
 7   Promo2SinceWeek            571 non-null    float64
 8   Promo2SinceYear            571 non-null    float64
 9   PromoInterval              571 non-null    object 
dtypes: float64(5), int64(2), object(3)
memory usage: 87.2+ KB


In [36]:
store.isnull().sum()

,0
Store,0
StoreType,0
Assortment,0
CompetitionDistance,3
CompetitionOpenSinceMonth,354
CompetitionOpenSinceYear,354
Promo2,0
Promo2SinceWeek,544
Promo2SinceYear,544
PromoInterval,544


In [37]:
# Get the exact null count and percentage for every column in store.csv

null_counts = store.isnull().sum()

null_pct = (null_counts / len(store) * 100).round(1)

pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})

,null_count,null_pct
Store,0,0.0
StoreType,0,0.0
Assortment,0,0.0
CompetitionDistance,3,0.3
CompetitionOpenSinceMonth,354,31.7
CompetitionOpenSinceYear,354,31.7
Promo2,0,0.0
Promo2SinceWeek,544,48.8
Promo2SinceYear,544,48.8
PromoInterval,544,48.8


Notice **Promo2SinceWeek, Promo2SinceYear, and PromoInterval** all show exactly 544 nulls — identical counts across all three.

That's not a coincidence; it's a strong signal they're all null for the same set of rows, which lines up exactly with our hypothesis: these should all be null precisely where **Promo2 == 0**.

Let's confirm this:

In [38]:
# Check: does the count of Promo2==0 rows match the 544 nulls exactly?
store['Promo2'].value_counts()

,count
Promo2,
1,571
0,544


In [39]:
# Direct test: are ALL 544 nulls in Promo2SinceWeek exactly the rows where Promo2 == 0?

nulls_match_promo2 = (store[store['Promo2SinceWeek'].isnull()]['Promo2'] == 0).all()

print(f"All Promo2SinceWeek nulls occur where Promo2==0: {nulls_match_promo2}")

All Promo2SinceWeek nulls occur where Promo2==0: True


In [40]:
# Cross-check: are there any Promo2==1 rows that still have a null in these fields? (would break the hypothesis)

broken_cases = store[(store['Promo2'] == 1) & (store['Promo2SinceWeek'].isnull())]

print(f"Promo2==1 rows with null Promo2SinceWeek (should be 0): {len(broken_cases)}")

Promo2==1 rows with null Promo2SinceWeek (should be 0): 0


Since they all are null where Promo2 == 0, we will leave this as is and handle this in the EDA part.

In [41]:
# Do the 3 CompetitionDistance nulls overlap with the 354 CompetitionOpenSinceMonth nulls, or are they distinct?

dist_null_rows = store[store['CompetitionDistance'].isnull()]

dist_null_rows[['Store','CompetitionDistance','CompetitionOpenSinceMonth','CompetitionOpenSinceYear']]

,Store,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear
290,291,NaN,NaN,NaN
621,622,NaN,NaN,NaN
878,879,NaN,NaN,NaN


In [42]:
# How many stores have a known CompetitionDistance but an unknown CompetitionOpenSinceMonth/Year?

known_dist_unknown_date = store[(store['CompetitionDistance'].notnull()) & (store['CompetitionOpenSinceMonth'].isnull())]

print(f"Stores with known distance but unknown competition open date: {len(known_dist_unknown_date)}")

Stores with known distance but unknown competition open date: 351


In [43]:
# Create a flag capturing whether competition opening date is known, before leaving the raw fields untouched

store['competition_open_date_known'] = store['CompetitionOpenSinceMonth'].notnull().astype(int)

In [44]:
# Verify: flag should be 0 for exactly 354 rows (the known nulls), 1 for the rest

store['competition_open_date_known'].value_counts()

,count
competition_open_date_known,
1,761
0,354


In [45]:
# Sanity check: flag should perfectly align with the null pattern in CompetitionOpenSinceMonth

mismatch = ((store['competition_open_date_known'] == 0) != store['CompetitionOpenSinceMonth'].isnull()).sum()

print(f"Mismatches between flag and actual nulls (should be 0): {mismatch}")

Mismatches between flag and actual nulls (should be 0): 0


In [46]:
# Checking for any duplicate values in the dataset:

store["Store"].nunique()

1115

In [47]:
# Check for fully duplicated rows (identical across all columns, not just Store ID)

print(f"Fully duplicate rows: {store.duplicated().sum()}")

Fully duplicate rows: 0


## Now, we have to merge train and store, Here's why:

**The core reason: these two tables answer different questions, but DemandLens needs both answered together for every prediction.**

`train.csv` is a **transactional/time-series table** — it answers "what happened on this specific day, at this specific store?" Each row is a (Store, Date) pair with that day's actual outcome: `Sales`, `Customers`, whether the store was `Open`, whether there was a `Promo` running, and holiday flags.

`store.csv` is a **reference/dimension table** — it answers "what kind of store is this, structurally?" Each row describes one store's relatively stable characteristics: its `StoreType`, `Assortment` strategy, how far away its nearest competitor is, and whether/when it participates in the recurring `Promo2` program.

Neither table alone tells the full story. `train.csv` on its own can tell you *that* Store 262 sold X amount on a given Thursday, but not *why* — is it a small `StoreType c` store with a competitor right next door, or a large `StoreType a` store with none nearby? Conversely, `store.csv` alone describes each store in the abstract, but has no sales data at all — it can't be used to forecast anything by itself.

**For a demand forecasting model, store-level context is a critical predictive signal, not a nice-to-have.** Two stores can show wildly different sales patterns not because of the date or the promo, but because of who they structurally are — their assortment type, their competitive environment, their historical promo participation. A model trained only on `train.csv` would be forecasting sales while blind to *why* different stores behave differently. Merging attaches that missing context to every single daily record, so each row a model sees during training carries both "what happened" and "what kind of store this is."

**Practically, the merge is a one-to-many join on `Store`.** Every store ID in `train.csv` appears on many rows (once per day it operated), while every store ID in `store.csv` appears exactly once (we just verified this — no duplicates, no missing IDs). Merging on `Store` broadcasts each store's static metadata onto every one of its daily transaction rows, so `StoreType`, `CompetitionDistance`, `Promo2`, and everything else in `store.csv` becomes available context for every single day in `train.csv`.

In [48]:
# Merge train (transactional) with store (reference) on Store ID — left join keeps every train row intact

train_merged = train.merge(store, on='Store', how='left')

In [49]:
# Validate: row count must stay exactly the same as train.csv — a left join should never add or drop rows

print(f"train.csv rows: {len(train)}, merged rows: {len(train_merged)}")

train.csv rows: 1017209, merged rows: 1017209


In [50]:
# Validate: check for any unmatched stores — nulls appearing in store.csv's columns would mean a Store ID in train had no match

print(train_merged[['StoreType','Assortment','CompetitionDistance']].isnull().sum())

StoreType                 0
Assortment                0
CompetitionDistance    2642
dtype: int64


In [51]:
# Verify: the 2,642 nulls in CompetitionDistance should exactly equal the total train rows for stores 291, 622, 879
null_dist_stores = [291, 622, 879]

expected_null_count = train[train['Store'].isin(null_dist_stores)].shape[0]

print(f"Expected CompetitionDistance nulls (rows for stores 291,622,879): {expected_null_count}")

print(f"Actual CompetitionDistance nulls in merged: {train_merged['CompetitionDistance'].isnull().sum()}")

Expected CompetitionDistance nulls (rows for stores 291,622,879): 2642
Actual CompetitionDistance nulls in merged: 2642


In [52]:
# Merge test with store on Store ID — same left join logic as train

test_merged = test.merge(store, on='Store', how='left')

# Validate: row count must stay exactly 41,088

print(f"test.csv rows: {len(test)}, merged rows: {len(test_merged)}")

test.csv rows: 41088, merged rows: 41088


In [58]:
train_merged.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,competition_open_date_known
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,1
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct",1
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct",1
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN,1
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN,1


In [59]:
test_merged.head()

,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,competition_open_date_known
0,1,1,4,2015-09-17,1.0,1,0,0,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,1
1,2,3,4,2015-09-17,1.0,1,0,0,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct",1
2,3,7,4,2015-09-17,1.0,1,0,0,a,c,24000.0,4.0,2013.0,0,NaN,NaN,NaN,1
3,4,8,4,2015-09-17,1.0,1,0,0,a,a,7520.0,10.0,2014.0,0,NaN,NaN,NaN,1
4,5,9,4,2015-09-17,1.0,1,0,0,a,c,2030.0,8.0,2000.0,0,NaN,NaN,NaN,1


In [53]:
# Validate: check for nulls appearing in store-originated columns (should be 0 for StoreType/Assortment, and the same 3-store pattern for CompetitionDistance)

print(test_merged[['StoreType','Assortment','CompetitionDistance']].isnull().sum())

StoreType               0
Assortment              0
CompetitionDistance    96
dtype: int64


In [54]:
# Final null audit — every column's null count in one view
print("=== NULL AUDIT (train_merged) ===")
null_summary = pd.DataFrame({
    'null_count': train_merged.isnull().sum(),
    'null_pct': (train_merged.isnull().sum() / len(train_merged) * 100).round(2)
})
print(null_summary[null_summary['null_count'] > 0])

# Range/sanity checks
print("\n=== SANITY CHECKS (train_merged) ===")
print(f"Sales min/max: {train_merged['Sales'].min()} / {train_merged['Sales'].max()}")
print(f"Open unique values: {train_merged['Open'].unique()}")
print(f"DayOfWeek range: {train_merged['DayOfWeek'].min()} to {train_merged['DayOfWeek'].max()}")
print(f"Date range: {train_merged['Date'].min()} to {train_merged['Date'].max()}")

# Confirm no duplicate (Store, Date) pairs — each store-day should be unique
dup_check = train_merged.duplicated(subset=['Store','Date']).sum()
print(f"\nDuplicate (Store, Date) pairs (should be 0): {dup_check}")

=== NULL AUDIT (train_merged) ===
                           null_count  null_pct
CompetitionDistance              2642      0.26
CompetitionOpenSinceMonth      323348     31.79
CompetitionOpenSinceYear       323348     31.79
Promo2SinceWeek                508031     49.94
Promo2SinceYear                508031     49.94
PromoInterval                  508031     49.94

=== SANITY CHECKS (train_merged) ===
Sales min/max: 0 / 41551
Open unique values: [1 0]
DayOfWeek range: 1 to 7
Date range: 2013-01-01 00:00:00 to 2015-07-31 00:00:00

Duplicate (Store, Date) pairs (should be 0): 0


In [56]:
# Save the cleaned, merged datasets to disk for reproducibility

train_merged.to_csv('train_cleaned.csv', index=False)

test_merged.to_csv('test_cleaned.csv', index=False)

In [57]:
# Quick confirmation they saved correctly

print(f"Saved train_cleaned.csv: {train_merged.shape}")

print(f"Saved test_cleaned.csv: {test_merged.shape}")

Saved train_cleaned.csv: (1017209, 19)
Saved test_cleaned.csv: (41088, 18)
